In [0]:
%sql
-- 1. Asset Hierarchy Master Table
CREATE TABLE IF NOT EXISTS dataworkspace.default.dim_asset_hierarchy (
    node_id STRING NOT NULL,
    node_name STRING NOT NULL,
    node_type STRING NOT NULL, 
    site_id STRING NOT NULL,
    manufacturer STRING,
    installation_date DATE,
    created_at TIMESTAMP
)
USING DELTA;

-- 2. Asset Relationships / Connectivity Table
CREATE TABLE IF NOT EXISTS dataworkspace.default.asset_relationship (
    relationship_id BIGINT GENERATED ALWAYS AS IDENTITY,
    parent_id STRING NOT NULL,
    child_id STRING NOT NULL,
    relationship_type STRING NOT NULL,
    is_active BOOLEAN
)
USING DELTA;

In [0]:
%sql
-- Insert Nodes
INSERT INTO dataworkspace.default.dim_asset_hierarchy 
(node_id, node_name, node_type, site_id, manufacturer, installation_date, created_at)
VALUES
('Site-A', 'Main Campus Site A', 'SITE', 'Site-A', NULL, NULL, current_timestamp()),
('Building-1', 'Operations Building 1', 'BUILDING', 'Site-A', NULL, NULL, current_timestamp()),
('Chiller-01', 'Centrifugal Chiller 01', 'EQUIPMENT', 'Site-A', 'Carrier', '2022-01-15', current_timestamp()),
('AHU-01', 'Air Handling Unit 01', 'EQUIPMENT', 'Site-A', 'Trane', '2022-02-10', current_timestamp()),
('AHU-02', 'Air Handling Unit 02', 'EQUIPMENT', 'Site-A', 'Trane', '2022-02-10', current_timestamp()),
('Pump-01', 'Chilled Water Pump 01', 'EQUIPMENT', 'Site-A', 'Grundfos', '2022-03-01', current_timestamp()),
('Flow Sensor-01', 'Water Flow Sensor 01', 'SENSOR', 'Site-A', 'Siemens', '2022-03-05', current_timestamp()),
('Orphan-Sensor-09', 'Detached Sensor 09', 'SENSOR', 'Site-A', 'Siemens', '2023-01-01', current_timestamp()),
('Isolated-Asset-99', 'Spare Motor 99', 'EQUIPMENT', 'Unknown', 'ABB', '2023-06-01', current_timestamp());

-- Insert Directed Relationships
INSERT INTO dataworkspace.default.asset_relationship 
(parent_id, child_id, relationship_type, is_active)
VALUES
('Site-A', 'Building-1', 'LOCATED_IN', true),
('Building-1', 'Chiller-01', 'HOUSES', true),
('Building-1', 'Pump-01', 'HOUSES', true),
('Chiller-01', 'AHU-01', 'FEEDS_TO', true),
('Chiller-01', 'AHU-02', 'FEEDS_TO', true),
('Pump-01', 'Flow Sensor-01', 'MONITORS', true);

num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql
select * from  dataworkspace.default.dim_asset_hierarchy 

node_id,node_name,node_type,site_id,manufacturer,installation_date,created_at
Site-A,Main Campus Site A,SITE,Site-A,null,null,2026-08-15T10:54:07.146Z
Building-1,Operations Building 1,BUILDING,Site-A,null,null,2026-08-15T10:54:07.146Z
Chiller-01,Centrifugal Chiller 01,EQUIPMENT,Site-A,Carrier,2022-01-15,2026-08-15T10:54:07.146Z
AHU-01,Air Handling Unit 01,EQUIPMENT,Site-A,Trane,2022-02-10,2026-08-15T10:54:07.146Z
AHU-02,Air Handling Unit 02,EQUIPMENT,Site-A,Trane,2022-02-10,2026-08-15T10:54:07.146Z
Pump-01,Chilled Water Pump 01,EQUIPMENT,Site-A,Grundfos,2022-03-01,2026-08-15T10:54:07.146Z
Flow Sensor-01,Water Flow Sensor 01,SENSOR,Site-A,Siemens,2022-03-05,2026-08-15T10:54:07.146Z
Orphan-Sensor-09,Detached Sensor 09,SENSOR,Site-A,Siemens,2023-01-01,2026-08-15T10:54:07.146Z
Isolated-Asset-99,Spare Motor 99,EQUIPMENT,Unknown,ABB,2023-06-01,2026-08-15T10:54:07.146Z


In [0]:
%sql
select * from dataworkspace.default.asset_relationship

relationship_id,parent_id,child_id,relationship_type,is_active
1,Site-A,Building-1,LOCATED_IN,true
2,Building-1,Chiller-01,HOUSES,true
3,Building-1,Pump-01,HOUSES,true
4,Chiller-01,AHU-01,FEEDS_TO,true
5,Chiller-01,AHU-02,FEEDS_TO,true
6,Pump-01,Flow Sensor-01,MONITORS,true


In [0]:
%sql
WITH RECURSIVE site_hierarchy AS (
    
    SELECT child_id AS asset_id, 1 AS depth
    FROM dataworkspace.default.asset_relationship
    WHERE parent_id = 'Site-A'
    
    UNION ALL
    
   
    SELECT r.child_id, sh.depth + 1
    FROM dataworkspace.default.asset_relationship r
    JOIN site_hierarchy sh ON r.parent_id = sh.asset_id
)
SELECT DISTINCT 
    h.node_id, 
    h.node_name, 
    h.node_type, 
    sh.depth
FROM site_hierarchy sh
JOIN dataworkspace.default.dim_asset_hierarchy h ON sh.asset_id = h.node_id
ORDER BY sh.depth, h.node_type;

node_id,node_name,node_type,depth
Building-1,Operations Building 1,BUILDING,1
Pump-01,Chilled Water Pump 01,EQUIPMENT,2
Chiller-01,Centrifugal Chiller 01,EQUIPMENT,2
AHU-02,Air Handling Unit 02,EQUIPMENT,3
AHU-01,Air Handling Unit 01,EQUIPMENT,3
Flow Sensor-01,Water Flow Sensor 01,SENSOR,3


In [0]:
%sql
SELECT 
    'PARENT (Upstream)' AS relation_direction,
    parent_id AS connected_node,
    relationship_type
FROM dataworkspace.default.asset_relationship
WHERE child_id = 'Chiller-01'

UNION ALL

SELECT 
    'CHILD (Downstream)' AS relation_direction,
    child_id AS connected_node,
    relationship_type
FROM dataworkspace.default.asset_relationship
WHERE parent_id = 'Chiller-01';

relation_direction,connected_node,relationship_type
PARENT (Upstream),Building-1,HOUSES
CHILD (Downstream),AHU-01,FEEDS_TO
CHILD (Downstream),AHU-02,FEEDS_TO


In [0]:
%sql
SELECT 
    h.node_id, 
    h.node_name, 
    h.node_type, 
    h.site_id
FROM dataworkspace.default.dim_asset_hierarchy h
LEFT JOIN dataworkspace.default.asset_relationship r ON h.node_id = r.child_id
WHERE r.child_id IS NULL 
  AND h.node_type != 'SITE';

node_id,node_name,node_type,site_id
Orphan-Sensor-09,Detached Sensor 09,SENSOR,Site-A
Isolated-Asset-99,Spare Motor 99,EQUIPMENT,Unknown


In [0]:
%sql
SELECT 
    h.node_id, 
    h.node_name, 
    h.node_type, 
    h.site_id
FROM dataworkspace.default.dim_asset_hierarchy h
LEFT JOIN dataworkspace.default.asset_relationship r_child ON h.node_id = r_child.parent_id
LEFT JOIN dataworkspace.default.asset_relationship r_parent ON h.node_id = r_parent.child_id
WHERE r_child.parent_id IS NULL 
  AND r_parent.child_id IS NULL 
  AND h.node_type != 'SITE';

node_id,node_name,node_type,site_id
Orphan-Sensor-09,Detached Sensor 09,SENSOR,Site-A
Isolated-Asset-99,Spare Motor 99,EQUIPMENT,Unknown
